# Phase 6 (Abstractive Summarization) — Run #3: Phase-1 **Decoder-only model 01** (12-layer MHA decoder, NoPE as trained) with **prefix tuning**

Config-driven: edit **Cell 2** only. Same data pipeline, chunk pairs, decoding sweep (incl. BERTScore selection, `min_new_all`, EOS threshold, proportional trimming), evaluation and submission as runs #1–#2, so all runs are directly comparable. Only the backbone and the adaptation method change. **One full run gives the final submission** (the decoding sweep tunes itself on VAL).

### Prefix tuning (Module 4.1 G; Li & Liang 2021)
The Phase-1 model is **completely frozen**. What is learned is a sequence of `P` virtual tokens that is prepended **at every layer**, as extra keys and values that every real token can attend to:

`head_i = softmax( q_i [K_prefix ; K_x]ᵀ / √d ) [V_prefix ; V_x]`

Each layer ℓ has its own `K_prefix^ℓ, V_prefix^ℓ ∈ R^{P × d}`. Unlike prompt tuning (which only adds input embeddings), the prefix steers every layer directly, which is why Li & Liang found it close to full fine-tuning on summarisation (XSum) with ~0.1 % of the weights.

**Implementation.** The Phase-1 code already supports a KV cache, so the prefix *is* a past KV cache of length `P` passed to every forward pass. Training (right padding, no mask) and generation (left padding, mask extended by `P` ones) both use the Phase-1 masking code unchanged.

**Parameterisation and initialisation** (both from the paper, Sec. 4.3 and 7.2):
* **Reparameterisation**: optimising the prefix directly is unstable, so `prefix = P₀ + MLP(E)`, with `E ∈ R^{P×d}` and an MLP `d → h → L·2·d`. After training only the resulting prefix (`L·2·P·d` numbers) is needed.
* **Initialisation from real words**: `P₀` is the KV cache of a real legal sentence of `P` tokens (computed once by the frozen model), and the MLP's last layer starts at **zero**. So at step 0 the model behaves exactly as if that sentence preceded the input (checked: identical to running the Phase-1 model on `sentence + input`). The paper reports that real-word initialisation clearly beats random initialisation, especially in low-data settings.

Defaults: `P = 64`, MLP hidden `h = 128`, so ~1.7M weights are trained (in the range of runs #1–#2, ~2M). The deployed prefix is 64 × 12 × 2 × 512 = 0.79M numbers.

**NoPE note.** This checkpoint has no positional encoding (it infers order from the causal mask), so the prefix needs no positions. The model never saw a prefix in pretraining, which is exactly what the reparameterised prefix learns to handle. Runs #1–#2 came from RoPE models; this run also tests whether a position-free decoder adapts as well.

### Run it
`SMOKE_TEST=True` first (~15 min: loss must fall, samples should read like legal English), then the full run. Paste the **"PASTE TO CLAUDE"** block.

In [1]:
# Cell 0 — Hugging Face login (token from Kaggle secrets), same as the Phase-1 notebooks
import os
try:
    from huggingface_hub import login
    from kaggle_secrets import UserSecretsClient
    _tok = UserSecretsClient().get_secret("HF")
    os.environ["HF_TOKEN"] = _tok
    login(_tok)
    print("HF login ok")
except Exception as e:
    print("HF secret not available:", e)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


HF login ok


In [2]:
# Cell 1 — Imports, paths, device, precision (same conventions as the Phase-1 notebooks)
import os, sys, json, math, time, random, gc, zipfile, types, re, pickle, copy, hashlib, bisect, subprocess
from pathlib import Path
from dataclasses import dataclass, asdict, field
from typing import Optional, Dict, List, Tuple, Any
from collections import Counter, defaultdict

import multiprocessing as mp
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint

IS_KAGGLE = Path("/kaggle/working").exists()
WORK_DIR = Path("/kaggle/working") if IS_KAGGLE else Path("./work")
WORK_DIR.mkdir(parents=True, exist_ok=True)
TOKENIZER_DIR = Path("/kaggle/input/datasets/ascyrax/capstone-tokenizer-large") if IS_KAGGLE else Path("./input")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    cap = torch.cuda.get_device_capability(0)
    AMP_DTYPE = torch.bfloat16 if cap[0] >= 8 else torch.float16     # T4/P100 -> fp16 + GradScaler
    USE_AMP = True
    print("GPU:", torch.cuda.get_device_name(0), "| capability", cap)
else:
    AMP_DTYPE, USE_AMP = torch.float32, False
USE_SCALER = USE_AMP and AMP_DTYPE == torch.float16
print(f"device={device} | AMP dtype={AMP_DTYPE} | grad scaler={USE_SCALER} | torch {torch.__version__}")

SEED = 1337
def seed_everything(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if device.type == "cuda": torch.cuda.manual_seed_all(s)
seed_everything(SEED)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

GPU: Tesla T4 | capability (7, 5)
device=cuda | AMP dtype=torch.float16 | grad scaler=True | torch 2.11.0+cu128


In [3]:
# Cell 2 — RUN CONFIG  (the only cell you normally edit)

RUN_NAME   = "p6_dec01_prefix"
METHOD     = "prefix"         # prefix tuning (Li & Liang 2021): learned per-layer key/value prefix, Phase-1 model frozen
SMOKE_TEST = False            # True: ~150 steps on a small subset, a few VAL documents generated, NO submission / export

# ---- Phase-1 checkpoint (caps_decoder_only_architecture_01.ipynb) -------------------------------------------------------
CKPT_KIND = "legacy_nope"     # "legacy_nope": the checkpoint as actually trained (no positional encoding) | "new": a RoPE retrain
CKPT_PATH = Path("/kaggle/input/datasets/ascyrax/caps-dec-01-best/best.pt") if IS_KAGGLE else Path("./input/best.pt")

# ---- Data / split (audit) ------------------------------------------------------------------------
HF_REPO        = "Exploration-Lab/CS-781-Capstone"     # Phase_6_Summ/{Train/train.json, Test/test.json}
LOCAL_DATA_DIR = None
P6             = "Phase_6_Summ"
VAL_N, VAL_SALT, VAL_CHECKSUM_EXPECTED = 200, "p6-val-v1", "5720e140c147"
MIN_SUMM_WORDS, MAX_SUMM_WORDS = 20, 5000       # training pairs only (VAL is kept as in the audit)
MAX_FULL_CHARS = 150_000                        # longer documents (< 1 %) -> first 2/3 + last 1/3 of this many characters

# ---- Chunk pairs ---------------------------------------------------------------------------------
CHUNK_TOKENS = 512           # document tokens per chunk
TGT_MAX      = 384           # target tokens per chunk (audit: p90 277; longer targets are cut at a sentence boundary)
MIN_ALIGN_F1 = 0.2           # weaker alignments are not assigned to a chunk (audit: 1.9 % of summary sentences)
EMPTY_KEEP   = 0.5           # share of chunks WITHOUT aligned summary text kept as EOS-only examples (resampled every epoch)
TAG_TEMPLATE = "[part {i}/{n}] "
SEP_TEXT     = "\n\nSUMMARY:\n"
N_PROC       = 4

# ---- Training ------------------------------------------------------------------------------------
EPOCHS        = 4            # same sequences as run #1 (~25-30 min/epoch on a T4 there); check the >> speed line
MICRO_TOKENS  = 12288        # padded tokens per micro-batch (n_seqs x longest); lower it on OOM
STEP_SEQS     = 32           # sequences per optimiser step (gradient accumulation)
WARMUP_FRAC   = 0.03
GRAD_CLIP     = 1.0
GRAD_CKPT     = False
TIME_BUDGET_MIN = 400        # training stops after this (best epoch kept); leaves ~3 h for evaluation, test and export
HP = {"prefix": dict(lr=2e-3, wd=0.0, prefix_len=64, mlp_hidden=128,
                   init_text="The following is a summary of a judgment of the Supreme Court of India. It states the facts of the case, the questions of law, the arguments of the parties, the reasoning of the Court and its final decision. ")}
SELECT_BY          = "loss"  # "loss" (VAL teacher-forced loss) | "rouge" (quick ROUGE on VAL_GEN_DOCS_EPOCH docs)
VAL_GEN_DOCS_EPOCH = 40

# ---- Generation ----------------------------------------------------------------------------------
GEN = dict(max_chunks=16,        # audit: 16 x 512 reads 95.7 % of all tokens
           max_new=TGT_MAX, min_new_first=32,   # the first chunk must say something (no empty summaries)
           min_new_all=64,                      # [EOS] banned for this many tokens in EVERY chunk (run #1 sweep: 0 -> 79 % empty chunks)
           eos_threshold=0.0,                   # > 0: stop only if p(EOS) >= this (greedy otherwise stops whenever EOS is the single top token)
           no_repeat_ngram=4, rep_penalty=1.1, batch=64,
           len_factor=1.25, dedup=True,
           trim="proportional")                 # over budget: "proportional" (every chunk keeps the same share) | "tail" (cut the end)
# Each entry overrides GEN for one sweep run (run #1: min_new_all 0 / 32 / 64 -> R1 0.200 / 0.531 / 0.586, still rising at 64)
DECODE_SWEEP = [dict(min_new_all=64), dict(min_new_all=96), dict(min_new_all=128),
                dict(min_new_all=32, eos_threshold=0.5), dict(min_new_all=64, eos_threshold=0.5),
                dict(min_new_all=96, trim="tail"), dict(min_new_all=96, max_chunks=24), dict(min_new_all=96, len_factor=1.5)]
SWEEP_DOCS   = 100           # VAL docs used to pick the decoding config; 0 = skip
SWEEP_SELECT = "bertscore"   # "bertscore" (primary metric; needs a GPU) | "rouge" ((R1 + RL) / 2)
LEN_MODEL    = dict(a=4.538474, b=0.6144775, lo=232.0, hi=2002.1)       # audit power law (words)
RUN_BERTSCORE, BERTSCORE_MODEL = True, "roberta-large"                   # metric only, never used for training
BASELINES = dict(tfidf=dict(r1=0.5749, r2=0.3186, rL=0.3008, bs=0.8533), lead=dict(r1=0.5222, r2=0.2508, rL=0.2437, bs=0.8321),
                 oracle=dict(r1=0.786, r2=0.6042, rL=0.5341, bs=0.8855))

# ---- Decode-only mode: skip training, load this run's saved adapter weights ----------------------
RESUME_FROM = None           # e.g. Path("/kaggle/input/<dataset>/summ_p6_dec02_lorma.pt") (or best_trainable.pt) -> sweep, VAL, test

# ---- KD teacher export (run #3) -------------------------------------------------------------------
EXPORT_TEACHER    = False    # this run is not a KD teacher (run #1 / #2 are the candidates)
TEACHER_TOPK      = 8        # top-k logits per gold target token (+ logsumexp), all TR pairs
TEACHER_GEN_PAIRS = 15000    # teacher-generated summaries for this many TR chunks (sequence-level KD)
TEACHER_GEN_MIN   = 90       # time cap (minutes) for the generation part

# ---- Smoke test ----------------------------------------------------------------------------------
SMOKE_STEPS, SMOKE_TRAIN_DOCS, SMOKE_VAL_DOCS = 150, 400, 12

# ---- derived -------------------------------------------------------------------------------------
assert METHOD in HP and SELECT_BY in ("loss", "rouge")
BODY = CHUNK_TOKENS
RUN_DIR = WORK_DIR / "runs" / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
print(f"RUN {RUN_NAME} | METHOD={METHOD} {HP[METHOD]} | chunk {CHUNK_TOKENS} -> target <= {TGT_MAX} | empty keep {EMPTY_KEEP} | "
      f"epochs {EPOCHS} | micro {MICRO_TOKENS} tok, step {STEP_SEQS} seqs | smoke={SMOKE_TEST}")

RUN p6_dec01_prefix | METHOD=prefix {'lr': 0.002, 'wd': 0.0, 'prefix_len': 64, 'mlp_hidden': 128, 'init_text': 'The following is a summary of a judgment of the Supreme Court of India. It states the facts of the case, the questions of law, the arguments of the parties, the reasoning of the Court and its final decision. '} | chunk 512 -> target <= 384 | empty keep 0.5 | epochs 4 | micro 12288 tok, step 32 seqs | smoke=False


In [4]:
# Cell 3 — The shared byte-level BPE tokenizer (same as every Phase-1 model)
if str(TOKENIZER_DIR) not in sys.path:
    sys.path.append(str(TOKENIZER_DIR))
from bpe_tokenizer import BPETokenizer, load_tokenizer

tokenizer = load_tokenizer(str(TOKENIZER_DIR))
VOCAB_SIZE = len(tokenizer.id_to_bytes)
BOS_ID, EOS_ID, PAD_ID, UNK_ID = tokenizer.bos_id, tokenizer.eos_id, tokenizer.pad_id, tokenizer.unk_id
print("vocab_size:", VOCAB_SIZE, "| BOS", BOS_ID, "EOS", EOS_ID, "PAD", PAD_ID)

_s = "Hon'ble Mr. Justice R. K. Gauba — § 302 IPC, ₹ 5,000 “quoted”."
_ids, _offs = tokenizer.encode(_s, return_offsets=True)
assert tokenizer.decode(_ids) == _s, "tokenizer round-trip failed"
assert all(0 <= a <= b <= len(_s) for a, b in _offs) and _offs[0][0] == 0 and _offs[-1][1] == len(_s)
print("offset sanity ok |", len(_ids), "tokens for", len(_s), "chars")

# Guard against O(L^2) BPE on pathological chunks (long runs of dashes, dots, underscores, spaces) -- as in Phases 3-4.
_MAX_PRETOK_BYTES = 1000
_orig_encode_pretok = tokenizer._encode_pretok_bytes
def _encode_pretok_bytes_capped(bs):
    if len(bs) <= _MAX_PRETOK_BYTES:
        return _orig_encode_pretok(bs)
    out = []
    for i in range(0, len(bs), _MAX_PRETOK_BYTES):
        out.extend(_orig_encode_pretok(bs[i:i + _MAX_PRETOK_BYTES]))
    return out
tokenizer._encode_pretok_bytes = _encode_pretok_bytes_capped
_t0 = time.time(); _ids = tokenizer.encode("-" * 200_000)
assert tokenizer.decode(_ids) == "-" * 200_000
print(f"long-run guard ok: 200k-char run encoded in {time.time() - _t0:.1f}s")

vocab_size: 32000 | BOS 2 EOS 3 PAD 0
offset sanity ok | 28 tokens for 62 chars
long-run guard ok: 200k-char run encoded in 0.1s


## Phase-1 model code — copied **verbatim** from `caps_decoder_only_architecture_01.ipynb` (Cells 4-10)
Do not edit: the checkpoint only loads correctly if these classes are identical.

In [5]:
# Cell 4 — Model configuration (decoder-only model 01, corrected)
#
#   preset_rope_mha()  -> model 01 AS INTENDED: RoPE + MHA + RMSNorm pre-norm + SwiGLU (retrain)
#   preset_nope_mha()  -> model 01 AS ACTUALLY TRAINED: identical but with no positional encoding.
#                         The original notebook built RoPE but never passed it to attention.
#                         Used to load the old checkpoint faithfully (MODE = "reuse", Cell 16).

@dataclass
class ModelConfig:
    # --- shape ---
    vocab_size: int = 32000
    n_layer: int = 12
    d_model: int = 512
    n_head: int = 8
    n_kv_head: int = 8                # == n_head -> MHA; < n_head -> GQA
    head_dim: int = 64
    ff_mult: float = 8/3              # SwiGLU hidden = round64(ff_mult * d_model) = 1408
    d_ff: Optional[int] = None
    max_seq_len: int = 1024

    # --- components ---
    norm_type: str = "rmsnorm"        # "rmsnorm" | "layernorm"
    norm_eps: float = 1e-5
    pos_type: str = "rope"            # "rope" | "learned" | "none"
    rope_theta: float = 10000.0
    mixer_type: str = "attention"
    mlp_type: str = "swiglu"          # "swiglu" | "gelu"
    channel_mixer_type: str = "mlp"

    # --- misc ---
    bias: bool = False
    attn_dropout: float = 0.0
    resid_dropout: float = 0.0
    embd_dropout: float = 0.0
    tie_embeddings: bool = True
    init_std: float = 0.02
    use_sdpa: bool = True

    def __post_init__(self):
        assert self.d_model == self.n_head * self.head_dim, "d_model must equal n_head*head_dim"
        assert self.n_head % self.n_kv_head == 0, "n_head must be divisible by n_kv_head"
        assert self.pos_type in ("rope", "learned", "none")
        if self.d_ff is None:
            raw = int(self.ff_mult * self.d_model)
            self.d_ff = 64 * ((raw + 63) // 64)


def preset_rope_mha(vocab_size=None, max_seq_len=1024):
    return ModelConfig(vocab_size=vocab_size or VOCAB_SIZE, pos_type="rope", max_seq_len=max_seq_len)

def preset_nope_mha(vocab_size=None, max_seq_len=1024):
    return ModelConfig(vocab_size=vocab_size or VOCAB_SIZE, pos_type="none", max_seq_len=max_seq_len)

PRESETS = {"rope_mha": preset_rope_mha, "nope_mha": preset_nope_mha}
print(json.dumps(asdict(preset_rope_mha()), indent=2))

{
  "vocab_size": 32000,
  "n_layer": 12,
  "d_model": 512,
  "n_head": 8,
  "n_kv_head": 8,
  "head_dim": 64,
  "ff_mult": 2.6666666666666665,
  "d_ff": 1408,
  "max_seq_len": 1024,
  "norm_type": "rmsnorm",
  "norm_eps": 1e-05,
  "pos_type": "rope",
  "rope_theta": 10000.0,
  "mixer_type": "attention",
  "mlp_type": "swiglu",
  "channel_mixer_type": "mlp",
  "bias": false,
  "attn_dropout": 0.0,
  "resid_dropout": 0.0,
  "embd_dropout": 0.0,
  "tie_embeddings": true,
  "init_std": 0.02,
  "use_sdpa": true
}


In [6]:
# Cell 5 — Normalisation layers (hand-written)

class LayerNorm(nn.Module):
    """Classic LayerNorm: centre by the mean, scale by the std, learn gain g and bias b."""
    def __init__(self, d: int, eps: float = 1e-5):
        super().__init__()
        self.g = nn.Parameter(torch.ones(d))
        self.b = nn.Parameter(torch.zeros(d))
        self.eps = eps

    def forward(self, x):
        xf = x.float()
        mean = xf.mean(-1, keepdim=True)
        var = xf.var(-1, keepdim=True, unbiased=False)
        return (((xf - mean) / torch.sqrt(var + self.eps)).to(x.dtype)) * self.g + self.b


class RMSNorm(nn.Module):
    """RMSNorm: no mean-centring and no bias; x / rms(x) * g. Statistics computed in fp32."""
    def __init__(self, d: int, eps: float = 1e-5):
        super().__init__()
        self.g = nn.Parameter(torch.ones(d))
        self.eps = eps

    def forward(self, x):
        rms = torch.sqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps)
        return (x.float() / rms).to(x.dtype) * self.g


def build_norm(cfg: ModelConfig, d: Optional[int] = None):
    d = d or cfg.d_model
    if cfg.norm_type == "layernorm":
        return LayerNorm(d, cfg.norm_eps)
    if cfg.norm_type == "rmsnorm":
        return RMSNorm(d, cfg.norm_eps)
    raise ValueError(cfg.norm_type)

In [7]:
# Cell 6 — Rotary position embeddings (RoPE)
#
# RoPE rotates each (even, odd) pair of q and k dimensions by angle p * theta^(-2i/D) at position p.
# Because rotations compose, q_i^T k_j depends only on the offset (i - j): relative position
# enters the attention score directly, with no parameters.
#
# FIX: in the original notebook this module was built but never reached the attention layers.
# Here the Backbone computes (cos, sin) once per forward and passes them to every layer.

class RoPE(nn.Module):
    """Table built for 2x the training length so generation never overflows it."""
    def __init__(self, head_dim: int, max_pos: int, theta: float = 10000.0):
        super().__init__()
        assert head_dim % 2 == 0
        inv_freq = 1.0 / (theta ** (torch.arange(0, head_dim, 2).float() / head_dim))
        freqs = torch.outer(torch.arange(max_pos).float(), inv_freq)   # [P, D/2]
        self.register_buffer("cos", freqs.cos(), persistent=False)
        self.register_buffer("sin", freqs.sin(), persistent=False)

    def tables(self, position_ids: torch.Tensor):
        """position_ids [B, T] -> (cos, sin) each [B, 1, T, D/2]."""
        return self.cos[position_ids].unsqueeze(1), self.sin[position_ids].unsqueeze(1)

    @staticmethod
    def apply(x, cos, sin):
        """x [B, H, T, D]; rotation in fp32 for stability."""
        dt = x.dtype
        x = x.float()
        x1, x2 = x[..., 0::2], x[..., 1::2]
        out = torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1).flatten(-2)
        return out.to(dt)

In [8]:
# Cell 7 — Shared attention kernel + standard MHA/GQA token mixer
#
# The Backbone builds ONE additive mask per forward pass (`attn_bias`, shape [B or 1, H or 1, T, S])
# containing causal + padding, and passes it to every layer. Every token mixer then
# calls `attend`. When attn_bias is None we use SDPA's fast is_causal path.

def attend(q, k, v, attn_bias, is_causal, scale, dropout_p=0.0, use_sdpa=True):
    """q [B,H,T,Dq], k [B,H,S,Dq], v [B,H,S,Dv] -> [B,H,T,Dv]."""
    if attn_bias is not None:
        attn_bias = attn_bias.to(q.dtype)          # SDPA requires mask dtype == query dtype
        is_causal = False
    if use_sdpa:
        return F.scaled_dot_product_attention(q, k, v, attn_mask=attn_bias,
                                              dropout_p=dropout_p, is_causal=is_causal,
                                              scale=scale)
    # Manual reference path (used by the tests to cross-check SDPA).
    scores = (q.float() @ k.float().transpose(-2, -1)) * scale
    if is_causal:
        T, S = q.shape[-2], k.shape[-2]
        scores = scores.masked_fill(
            torch.triu(torch.ones(T, S, dtype=torch.bool, device=q.device), diagonal=S - T + 1),
            float("-inf"))
    if attn_bias is not None:
        scores = scores + attn_bias.float()
    p = F.softmax(scores, dim=-1)
    if dropout_p > 0:
        p = F.dropout(p, dropout_p)
    return (p @ v.float()).to(q.dtype)


class Attention(nn.Module):
    """Multi-head (n_kv_head == n_head) or grouped-query (n_kv_head < n_head) self-attention.
    Cache format: (k, v), each [B, n_kv_head, S, head_dim]."""
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg
        self.n_head, self.n_kv_head, self.head_dim = cfg.n_head, cfg.n_kv_head, cfg.head_dim
        self.scale = 1.0 / math.sqrt(cfg.head_dim)
        self.q_proj = nn.Linear(cfg.d_model, cfg.n_head * cfg.head_dim, bias=cfg.bias)
        self.k_proj = nn.Linear(cfg.d_model, cfg.n_kv_head * cfg.head_dim, bias=cfg.bias)
        self.v_proj = nn.Linear(cfg.d_model, cfg.n_kv_head * cfg.head_dim, bias=cfg.bias)
        self.o_proj = nn.Linear(cfg.n_head * cfg.head_dim, cfg.d_model, bias=cfg.bias)
        self.resid_dropout = nn.Dropout(cfg.resid_dropout)

    def _split(self, x, n):
        B, T, _ = x.shape
        return x.view(B, T, n, self.head_dim).transpose(1, 2)

    @staticmethod
    def cache_len(cache):
        return cache[0].shape[2]

    def cache_numbers_per_token(self):
        return 2 * self.n_kv_head * self.head_dim

    def forward(self, x, attn_bias=None, rope_cs=None, kv_cache=None):
        B, T, _ = x.shape
        q = self._split(self.q_proj(x), self.n_head)
        k = self._split(self.k_proj(x), self.n_kv_head)
        v = self._split(self.v_proj(x), self.n_kv_head)
        if rope_cs is not None:                       # FIX: RoPE is now actually applied
            cos, sin = rope_cs
            q, k = RoPE.apply(q, cos, sin), RoPE.apply(k, cos, sin)
        if kv_cache is not None:
            k = torch.cat([kv_cache[0], k], dim=2)
            v = torch.cat([kv_cache[1], v], dim=2)
        new_cache = (k, v)
        if self.n_kv_head != self.n_head:
            rep = self.n_head // self.n_kv_head
            k, v = k.repeat_interleave(rep, dim=1), v.repeat_interleave(rep, dim=1)
        out = attend(q, k, v, attn_bias, is_causal=(attn_bias is None and T > 1),
                     scale=self.scale,
                     dropout_p=self.cfg.attn_dropout if self.training else 0.0,
                     use_sdpa=self.cfg.use_sdpa)
        out = out.transpose(1, 2).contiguous().view(B, T, -1)
        return self.resid_dropout(self.o_proj(out)), new_cache

In [9]:
# Cell 8 — Channel mixers, registries, transformer block

class SwiGLU(nn.Module):
    """down( silu(gate(x)) * up(x) )."""
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.gate_proj = nn.Linear(cfg.d_model, cfg.d_ff, bias=cfg.bias)
        self.up_proj = nn.Linear(cfg.d_model, cfg.d_ff, bias=cfg.bias)
        self.down_proj = nn.Linear(cfg.d_ff, cfg.d_model, bias=cfg.bias)
        self.drop = nn.Dropout(cfg.resid_dropout)

    def forward(self, x):
        return self.drop(self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x)))


class GELUMLP(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.fc1 = nn.Linear(cfg.d_model, cfg.d_ff, bias=cfg.bias)
        self.down_proj = nn.Linear(cfg.d_ff, cfg.d_model, bias=cfg.bias)
        self.drop = nn.Dropout(cfg.resid_dropout)

    def forward(self, x):
        return self.drop(self.down_proj(F.gelu(self.fc1(x))))


TOKEN_MIXERS: Dict[str, Any] = {}
CHANNEL_MIXERS: Dict[str, Any] = {}

def register_token_mixer(name):
    def deco(fn):
        TOKEN_MIXERS[name] = fn
        return fn
    return deco

def register_channel_mixer(name):
    def deco(fn):
        CHANNEL_MIXERS[name] = fn
        return fn
    return deco

# FIX: token mixers no longer receive `rope` at build time (model 01 silently dropped it there).
# Positional information now flows through forward(): rope_cs / attn_bias, built in the Backbone.
@register_token_mixer("attention")
def _build_attention(cfg):
    return Attention(cfg)

@register_channel_mixer("mlp")
def _build_mlp(cfg):
    return {"swiglu": SwiGLU, "gelu": GELUMLP}[cfg.mlp_type](cfg)


class Block(nn.Module):
    """Pre-norm block: x = x + mixer(norm(x)); x = x + mlp(norm(x))."""
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.norm1 = build_norm(cfg)
        self.mixer = TOKEN_MIXERS[cfg.mixer_type](cfg)
        self.norm2 = build_norm(cfg)
        self.cmix = CHANNEL_MIXERS[cfg.channel_mixer_type](cfg)

    def forward(self, x, **kw):
        h, new_cache = self.mixer(self.norm1(x), **kw)
        x = x + h
        x = x + self.cmix(self.norm2(x))
        return x, new_cache

In [10]:
# Cell 9 — Backbone and language model

class Backbone(nn.Module):
    """Embedding -> N blocks -> final norm. Builds positional info and masks ONCE per forward."""
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.emb_drop = nn.Dropout(cfg.embd_dropout)
        self.rope = RoPE(cfg.head_dim, 2 * cfg.max_seq_len, cfg.rope_theta) \
            if cfg.pos_type == "rope" else None
        self.pos_emb = nn.Embedding(2 * cfg.max_seq_len, cfg.d_model) \
            if cfg.pos_type == "learned" else None
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layer)])
        self.norm_f = build_norm(cfg)

    def _attn_bias(self, B, T, past, attention_mask, device):
        """Additive mask [B|1, H|1, T, S] or None (=> SDPA fast causal path)."""
        S = past + T
        need = (attention_mask is not None) or (past > 0 and T > 1)
        if not need:
            return None
        q_pos = torch.arange(past, S, device=device)
        k_pos = torch.arange(S, device=device)
        bias = torch.zeros(T, S, device=device).masked_fill(
            k_pos[None, :] > q_pos[:, None], float("-inf"))[None, None]          # causal
        if attention_mask is not None:
            # Mask padded KEYS. A padded QUERY would then have every key masked -> softmax of
            # all -inf -> NaN, which can leak into later layers through 0*NaN. So let each
            # padded query attend to itself (its output is never used).
            key_pad = ~attention_mask.bool()                                   # [B, S]
            q_idx = torch.arange(past, S, device=device)
            k_idx = torch.arange(S, device=device)
            is_self = q_idx[:, None] == k_idx[None, :]                         # [T, S]
            block = key_pad[:, None, :] & ~is_self[None]                       # [B, T, S]
            pad = torch.zeros(B, 1, T, S, device=device).masked_fill(block[:, None], float("-inf"))
            bias = bias + pad
        return bias

    def forward(self, input_ids, attention_mask=None, kv_caches=None):
        B, T = input_ids.shape
        dev = input_ids.device
        past = 0 if kv_caches is None else self.blocks[0].mixer.cache_len(kv_caches[0])
        x = self.tok_emb(input_ids)

        rope_cs = None
        if self.rope is not None or self.pos_emb is not None:
            if attention_mask is not None:       # left padding: real tokens start at position 0
                pos = (attention_mask.long().cumsum(-1) - 1).clamp(min=0)[:, -T:]
            else:
                pos = torch.arange(past, past + T, device=dev).unsqueeze(0).expand(B, T)
            if self.rope is not None:
                rope_cs = self.rope.tables(pos)
            else:
                x = x + self.pos_emb(pos)

        attn_bias = self._attn_bias(B, T, past, attention_mask, dev)
        x = self.emb_drop(x)
        new_caches = []
        for i, blk in enumerate(self.blocks):
            x, nc = blk(x, attn_bias=attn_bias, rope_cs=rope_cs,
                        kv_cache=None if kv_caches is None else kv_caches[i])
            new_caches.append(nc)
        return self.norm_f(x), new_caches


class LegalLM(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg
        self.backbone = Backbone(cfg)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        if cfg.tie_embeddings:
            self.lm_head.weight = self.backbone.tok_emb.weight

    def forward(self, input_ids, targets=None, attention_mask=None, kv_caches=None,
                last_only=False, reduction="mean"):
        h, new_caches = self.backbone(input_ids, attention_mask=attention_mask,
                                      kv_caches=kv_caches)
        if last_only:                 # generation: only the last position's logits are needed
            h = h[:, -1:, :]
        logits = self.lm_head(h)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.float().view(-1, logits.size(-1)), targets.reshape(-1),
                                   ignore_index=-100, reduction=reduction)
            if reduction == "none":
                loss = loss.view(targets.shape)
        return logits, loss, new_caches

In [11]:
# Cell 10 — Init, parameter counting, optimiser groups, checkpoint I/O

def init_weights(model: LegalLM, cfg: ModelConfig):
    """N(0, 0.02) everywhere; residual-branch output projections scaled by 1/sqrt(2L)
    so the residual stream's variance does not grow with depth."""
    for m in model.modules():
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, 0.0, cfg.init_std)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, 0.0, cfg.init_std)
    scale = 1.0 / math.sqrt(2 * cfg.n_layer)
    with torch.no_grad():
        for blk in model.backbone.blocks:
            blk.mixer.o_proj.weight.mul_(scale)
            blk.cmix.down_proj.weight.mul_(scale)


def count_params(model: LegalLM) -> Dict[str, int]:
    seen, total = set(), 0
    for p in model.parameters():
        if id(p) not in seen:
            seen.add(id(p))
            total += p.numel()
    emb = model.backbone.tok_emb.weight.numel()
    attn = sum(p.numel() for blk in model.backbone.blocks for p in blk.mixer.parameters())
    return {"total": total, "embedding": emb, "non_embedding": total - emb, "attention": attn}


def build_param_groups(model: nn.Module, weight_decay: float):
    """Decay only 2-D weight matrices (not embeddings, norm gains or biases)."""
    decay, no_decay = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if p.ndim >= 2 and "tok_emb" not in name and "pos_emb" not in name:
            decay.append(p)
        else:
            no_decay.append(p)
    return [{"params": decay, "weight_decay": weight_decay},
            {"params": no_decay, "weight_decay": 0.0}]


def save_model(path, model: LegalLM, cfg: ModelConfig, extra: dict = None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save({"model": model.state_dict(), "config": asdict(cfg), "extra": extra or {}}, path)


def load_model(path, map_location="cpu") -> Tuple[LegalLM, ModelConfig, dict]:
    ckpt = torch.load(path, map_location=map_location, weights_only=False)
    cfg = ModelConfig(**ckpt["config"])
    model = LegalLM(cfg)
    model.load_state_dict(ckpt["model"])
    return model, cfg, ckpt.get("extra", {})


def load_legacy_model01(path, map_location="cpu") -> Tuple[LegalLM, ModelConfig, dict]:
    """Load a checkpoint written by the ORIGINAL decoder-only notebook.
    Parameter names are unchanged, so the weights load strictly. The config is rebuilt with
    pos_type="none" because that is what was actually trained (RoPE was never applied).
    Loading it as "rope" would silently change the function the weights compute."""
    ckpt = torch.load(path, map_location=map_location, weights_only=False)
    old = ckpt["config"]
    kept = {k: v for k, v in old.items() if k in ModelConfig.__dataclass_fields__}
    dropped = sorted(set(old) - set(kept))
    kept["pos_type"] = "none"
    cfg = ModelConfig(**kept)
    model = LegalLM(cfg)
    model.load_state_dict(ckpt["model"], strict=True)
    print(f"legacy model 01 loaded as NoPE | dropped old config fields: {dropped}")
    return model, cfg, ckpt.get("extra", {})

## Load the Phase-1 checkpoint

In [12]:
# Cell 11 — Load the Phase-1 checkpoint and sanity-check it
if CKPT_KIND == "legacy_nope":
    lm, lm_cfg, lm_extra = load_legacy_model01(CKPT_PATH, map_location="cpu")    # loads as NoPE, as it was trained
else:
    lm, lm_cfg, lm_extra = load_model(CKPT_PATH, map_location="cpu")
lm = lm.to(device).eval()
pc = count_params(lm)
print(f"Phase-1 model | pos_type={lm_cfg.pos_type} | layers={lm_cfg.n_layer} d_model={lm_cfg.d_model} | "
      f"params total={pc['total']/1e6:.2f}M (embedding {pc['embedding']/1e6:.2f}M)")
if lm_extra.get("val_loss") is not None:
    print("logged Phase-1 val loss:", lm_extra["val_loss"])

# If the weights did not load properly, the LM loss on a normal legal sentence would sit near ln(V) ~ 10.4.
_sent = "The appeal is allowed and the impugned judgment of the High Court is set aside."
_ids = [BOS_ID] + tokenizer.encode(_sent)
_x = torch.tensor([_ids[:-1]], device=device); _y = torch.tensor([_ids[1:]], device=device)
with torch.no_grad(), torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
    _, _l, _ = lm(_x, targets=_y)
print(f"LM loss on a legal sentence: {_l.item():.3f}  (ln V = {math.log(VOCAB_SIZE):.2f})")
assert _l.item() < 9.0, "checkpoint looks untrained / mis-loaded"

assert VOCAB_SIZE < 65536, "token ids are stored as uint16"
print("attention: causal (Phase-1 mask) | positional encoding:", lm_cfg.pos_type)

legacy model 01 loaded as NoPE | dropped old config fields: ['causal', 'grad_checkpointing']
Phase-1 model | pos_type=none | layers=12 d_model=512 | params total=54.93M (embedding 16.38M)
logged Phase-1 val loss: 2.116826470941305
LM loss on a legal sentence: 2.015  (ln V = 10.37)
attention: causal (Phase-1 mask) | positional encoding: none


## Phase-6 data, the fixed validation split, cleaning
`split_train` and `_h` are copied **verbatim** from the audit (Cell 5); the checksum assert guarantees that every Phase-6 run is scored on the same 200 documents. Cleaning (too-short / too-long summaries, summaries longer than their document) is applied to the **training** part only.

In [13]:
# Cell 13 — Load train / test, split, clean
from huggingface_hub import hf_hub_download
from tqdm.auto import tqdm

def fetch(rel):
    if LOCAL_DATA_DIR is not None and (Path(LOCAL_DATA_DIR) / rel).exists():
        return Path(LOCAL_DATA_DIR) / rel
    try:
        return Path(hf_hub_download(repo_id=HF_REPO, repo_type="dataset", filename=rel, token=os.environ.get("HF_TOKEN")))
    except Exception as ex:
        raise RuntimeError(f"could not download '{rel}' from {HF_REPO} ({type(ex).__name__}: {ex}). "
                           f"Check the HF login (Cell 0) / internet, or download {P6} and set LOCAL_DATA_DIR.") from ex

ID_KEYS  = ("id", "doc_id", "document_id", "case_id")
DOC_KEYS = ("document", "text", "judgment", "judgement", "source", "case_text", "content")
SUM_KEYS = ("summary", "reference_summary", "abstract", "target", "headnote")
def _first(e, keys, default=None):
    for k in keys:
        if k in e: return e[k]
    return default
def _as_text(v): return "\n".join(str(t) for t in v) if isinstance(v, list) else v

def _read_json_any(path):
    txt = Path(path).read_text(encoding="utf-8")
    try: return json.loads(txt)
    except json.JSONDecodeError: return [json.loads(l) for l in txt.splitlines() if l.strip()]

def parse_docs(raw, split, has_sum):
    if isinstance(raw, dict):
        if isinstance(raw.get("data"), list): items = raw["data"]
        else:
            items = []
            for k, v in raw.items():
                v = dict(v) if isinstance(v, dict) else {"document": v}
                v.setdefault("id", k); items.append(v)
    else:
        items = list(raw)
    docs = []
    for i, e in enumerate(items):
        rid = _first(e, ID_KEYS, i)
        doc = _as_text(_first(e, DOC_KEYS))
        if not isinstance(doc, str): raise ValueError(f"{split} item {rid}: no document field. Keys: {list(e)}")
        summ = _as_text(_first(e, SUM_KEYS)) if has_sum else None
        if has_sum and not isinstance(summ, str): raise ValueError(f"{split} item {rid}: no summary field. Keys: {list(e)}")
        docs.append(dict(uid=f"{split}::{i}", id=rid, doc=doc, summ=summ))
    return docs

def load_split(rel, split, has_sum):
    raw = _read_json_any(fetch(rel))
    print(f"  {rel}: {type(raw).__name__} of {len(raw)}")
    return parse_docs(raw, split, has_sum)

TRAIN = load_split(f"{P6}/Train/train.json", "train", True)
TEST_ALL = load_split(f"{P6}/Test/test.json", "test", False)

# ---- verbatim from the audit (Cell 5) ----
def _norm(t): return re.sub(r"\s+", " ", t).strip().lower()
def _h(t):    return hashlib.md5(_norm(t).encode("utf-8")).hexdigest()
def split_train(train, val_n=VAL_N, salt=VAL_SALT):
    # Whole duplicate-document groups go to one side; group order = md5(salt :: normalised-text hash).
    groups = defaultdict(list)
    for d in train: groups[_h(d["doc"])].append(d)
    order = sorted(groups, key=lambda g: hashlib.md5(f"{salt}::{g}".encode()).hexdigest())
    val, val_keys = [], set()
    for g in order:
        if len(val) >= val_n: break
        val.extend(groups[g]); val_keys.add(g)
    tr = [d for d in train if _h(d["doc"]) not in val_keys]
    checksum = hashlib.md5("|".join(sorted(str(d["id"]) for d in val)).encode()).hexdigest()[:12]
    return tr, val, checksum
# ------------------------------------------

TR_ALL, VAL_ALL, _cs = split_train(TRAIN)
print(f"split: TR {len(TR_ALL)} | VAL {len(VAL_ALL)} | checksum {_cs}")
assert _cs == VAL_CHECKSUM_EXPECTED, f"VAL split differs from the audit ({_cs} != {VAL_CHECKSUM_EXPECTED})"

def _clean(docs):
    keep, drop = [], Counter()
    seen = set()
    for d in docs:
        sw, dw = len(d["summ"].split()), len(d["doc"].split())
        key = (_h(d["doc"]), _h(d["summ"]))
        if sw < MIN_SUMM_WORDS: drop["short summary"] += 1; continue
        if sw > MAX_SUMM_WORDS: drop["very long summary"] += 1; continue
        if sw > dw: drop["summary longer than document"] += 1; continue
        if key in seen: drop["exact duplicate pair"] += 1; continue
        seen.add(key); keep.append(d)
    return keep, dict(drop)
TR_ALL, _dropped = _clean(TR_ALL)
print(f"training docs after cleaning: {len(TR_ALL)} | dropped {_dropped}")

Phase_6_Summ/Train/train.json: reconstructing file:   0%|          |  0.00B /  211MB            

Phase_6_Summ/Train/train.json: downloading bytes:           |  0.00B            

  Phase_6_Summ/Train/train.json: list of 7030


test.json:   0%|          | 0.00/2.73M [00:00<?, ?B/s]

  Phase_6_Summ/Test/test.json: list of 100
split: TR 6830 | VAL 200 | checksum 5720e140c147
training docs after cleaning: 6759 | dropped {'exact duplicate pair': 25, 'very long summary': 43, 'summary longer than document': 1, 'short summary': 2}


## Sentence alignment → chunk-level training pairs
`split_sentences`, `content` and `align_doc` are the audit's (Cell 8), unchanged. For every training document: tokenise it **with character offsets**, cut it into 512-token chunks, align each summary sentence to its best document sentence, and give the sentence to the chunk containing that source sentence. A chunk's target is its sentences **in summary order**, cut at a sentence boundary to `TGT_MAX` tokens (a single over-long sentence is cut and gets no `[EOS]`, so the model never learns to stop mid-sentence). Tokenisation and alignment run in `N_PROC` processes and are cached.

In [14]:
# Cell 14 — Sentence splitter + aligner (verbatim from the audit, Cell 8)
_WRE = re.compile(r"[a-z0-9]+")
def words(t): return _WRE.findall(t.lower())
_ABBR = (r"mr|mrs|ms|dr|hon'ble|honble|no|nos|sr|jr|ltd|pvt|co|corp|inc|vs|v|viz|i\.e|e\.g|etc|cf|art|arts|sec|secs|s|ss|cl|"
         r"para|paras|p|pp|vol|ors|anr|govt|st|rs|prof|smt|shri|sri|ch|app|crl|cr|civ|j|jj|cj|mad|bom|cal|del|all|scc|scr|air|"
         r"u/s|r/w|w\.e\.f|addl|asstt|dist|distt|dept|reg|regd|ft|sq|nos|approx|ibid|supra|id|cri|lj|cri\.l\.j|misc|wp|slp")
_ABBR_END = re.compile(r"(?:\b(?:" + _ABBR + r")|\b[A-Za-z])\.$|^\s*\(?\d{1,3}[.)]$", re.I)
_SPLIT = re.compile(r"(?<=[.?!])[\"')\]]*\s+(?=[\"'(\[]?[A-Z0-9])|\n\s*\n|\n(?=\s*(?:\d{1,3}[.)]\s|\([a-z0-9]{1,4}\)\s))")

def split_sentences(text):
    # Returns [(char_start, sentence)]. Pieces ending in an abbreviation / initial / short number are merged forward.
    out, start = [], 0
    pieces = []
    for m in _SPLIT.finditer(text):
        pieces.append((start, text[start:m.start()])); start = m.end()
    pieces.append((start, text[start:]))
    buf_s, buf = None, ""
    for s, p in pieces:
        if buf_s is None: buf_s, buf = s, p
        else: buf = buf + " " + p
        if not _ABBR_END.search(buf.rstrip()):
            if buf.strip(): out.append((buf_s, buf.strip()))
            buf_s, buf = None, ""
    if buf_s is not None and buf.strip(): out.append((buf_s, buf.strip()))
    return out

_STOP = set(("the a an of to in and or for on by with at from as is was were be been are that this these those it its which who "
             "whom whose has have had not no but if than then so such any all also there their they he she his her him we our us you "
             "your i me my shall may can could would should will said upon into under over between after before against within "
             "without about being do does did same other further thereof therein herein hereby whereas").split())
def content(t): return {w for w in _WRE.findall(t.lower()) if w not in _STOP and len(w) > 1}

def align_doc(doc, summ):
    ds, ss = split_sentences(doc), split_sentences(summ)
    dsets = [content(t) for _, t in ds]
    inv = defaultdict(list)
    for j, st in enumerate(dsets):
        for w in st: inv[w].append(j)
    res = []
    for k, (_, t) in enumerate(ss):
        q = content(t)
        if len(q) < 2: continue
        hits = Counter()
        for w in q:
            for j in inv.get(w, ()): hits[j] += 1
        best_j, best_f = -1, 0.0
        for j, c in hits.items():
            f = 2 * c / (len(q) + len(dsets[j]))
            if f > best_f: best_j, best_f = j, f
        if best_j >= 0:
            res.append(dict(k=k, j=best_j, f=best_f, pos=ds[best_j][0] / max(len(doc), 1), char=ds[best_j][0], words=len(t.split())))
    return ds, ss, res

In [15]:
# Cell 15 — Tokenise documents into chunks (+ chunk targets for documents with a summary); cache; prompt pieces
def cut_text(t):
    if len(t) <= MAX_FULL_CHARS: return t
    h = 2 * MAX_FULL_CHARS // 3
    return t[:h] + "\n" + t[-(MAX_FULL_CHARS - h):]

def _fit_target(sents):
    # sentences (summary order) -> (token ids <= TGT_MAX, truncated?) ; whole sentences while they fit
    chosen, ids = [], []
    for s in sents:
        cand = tokenizer.encode(" ".join(chosen + [s]))
        if len(cand) <= TGT_MAX: chosen.append(s); ids = cand
        else: break
    if chosen: return ids, False
    return tokenizer.encode(sents[0])[:TGT_MAX], True

_DOCS_W, _WITH_TGT = None, False
def _work(i):
    d = _DOCS_W[i]
    text = cut_text(d["doc"])
    ids, offs = tokenizer.encode(text, return_offsets=True)
    ids = list(ids)
    if not ids: ids, offs = [EOS_ID], [(0, 0)]
    starts = list(range(0, len(ids), BODY))
    cchar = [offs[s][0] for s in starts]
    chunks = [np.asarray(ids[s:s + BODY], dtype=np.uint16) for s in starts]
    tgts, trunc = None, None
    if _WITH_TGT and d["summ"] is not None:
        _, ss, res = align_doc(text, d["summ"])
        per = defaultdict(set)
        for r in res:
            if r["f"] >= MIN_ALIGN_F1:
                per[max(0, bisect.bisect_right(cchar, r["char"]) - 1)].add(r["k"])
        tgts, trunc = [], []
        for j in range(len(starts)):
            if not per.get(j): tgts.append(None); trunc.append(False); continue
            t_ids, tr = _fit_target([ss[k][1] for k in sorted(per[j])])
            tgts.append(np.asarray(t_ids, dtype=np.uint16)); trunc.append(tr)
    return dict(chunks=chunks, tgts=tgts, trunc=trunc, ntok=len(ids), words=len(d["doc"].split()))

def prepare_docs(docs, cache_name, with_targets):
    global _DOCS_W, _WITH_TGT
    cdir = WORK_DIR / "cache"; cdir.mkdir(exist_ok=True)
    cp = cdir / f"p6_{cache_name}_v{VOCAB_SIZE}_c{CHUNK_TOKENS}_t{TGT_MAX}_a{MIN_ALIGN_F1}_m{MAX_FULL_CHARS}_{int(with_targets)}.pkl"
    cache = pickle.load(open(cp, "rb")) if cp.exists() else {}
    key = lambda d: (d["uid"], len(d["doc"]), len(d["summ"] or ""))
    todo = [i for i, d in enumerate(docs) if cache.get(d["uid"], (None,))[0] != key(d)]
    if todo:
        _DOCS_W, _WITH_TGT = docs, with_targets
        order = sorted(todo, key=lambda i: -len(docs[i]["doc"]))
        res = None
        if N_PROC > 1:
            try:
                with mp.get_context("fork").Pool(N_PROC) as pool:
                    res = list(tqdm(pool.imap(_work, order, chunksize=8), total=len(order), desc=f"prepare {cache_name}", leave=False))
            except Exception as ex:
                print("parallel preparation failed (", type(ex).__name__, ex, ") -> serial"); res = None
        if res is None:
            res = [_work(i) for i in tqdm(order, desc=f"prepare {cache_name}", leave=False)]
        for i, r in zip(order, res): cache[docs[i]["uid"]] = (key(docs[i]), r)
        pickle.dump(cache, open(cp, "wb"))
    for d in docs: d.update(cache[d["uid"]][1])
    return docs

SEP_IDS = tokenizer.encode(SEP_TEXT)
_TAG_CACHE = {}
def tag_ids(j, n):
    k = (j, n)
    if k not in _TAG_CACHE: _TAG_CACHE[k] = tokenizer.encode(TAG_TEMPLATE.format(i=j + 1, n=n))
    return _TAG_CACHE[k]
def prompt_ids(d, j):
    return [BOS_ID] + tag_ids(j, len(d["chunks"])) + d["chunks"][j].astype(np.int64).tolist() + SEP_IDS
MAX_PROMPT = 1 + len(tag_ids(998, 999)) + BODY + len(SEP_IDS)
assert MAX_PROMPT + TGT_MAX + 1 <= lm_cfg.max_seq_len, f"prompt {MAX_PROMPT} + target {TGT_MAX} exceeds the RoPE table {lm_cfg.max_seq_len}"
print(f"prompt <= {MAX_PROMPT} tokens + target <= {TGT_MAX} + EOS -> sequences <= {MAX_PROMPT + TGT_MAX + 1} (context {lm_cfg.max_seq_len})")

def pair_stats(docs, title):
    n_ch = np.asarray([len(d["chunks"]) for d in docs])
    t = [len(x) for d in docs for x in d["tgts"] if x is not None]
    emp = np.mean([x is None for d in docs for x in d["tgts"]])
    tr = np.mean([f for d in docs for x, f in zip(d["tgts"], d["trunc"]) if x is not None])
    print(f"{title}: {len(docs)} docs | chunks/doc mean {n_ch.mean():.1f} median {np.median(n_ch):.0f} | chunk pairs with a target {len(t)} "
          f"| empty chunks {100 * emp:.1f}% | target tokens median {np.median(t):.0f} p90 {np.percentile(t, 90):.0f} | "
          f"single-sentence cut {100 * tr:.1f}%")

prompt <= 529 tokens + target <= 384 + EOS -> sequences <= 914 (context 1024)


## Prefix, batches, loss, generation
* `prefix_kv(model, B)` returns the per-layer `(K, V)` prefix, each `[B, n_kv_head, P, head_dim]`, and is passed to the Phase-1 backbone as `kv_caches`. Gradients flow into the prefix parameters through the concatenation inside the attention layer.
* Training batches are right-padded with no padding mask (a causal query never sees later padding); the backbone builds the causal mask with the `P`-token offset itself.
* Generation: left-padded prompts, padding mask = `[1]·P ++ mask`, KV cache grows from the prefix. Same decoding controls as runs #1–#2. Cell 19 checks it against one-sequence, no-cache greedy decoding **with the prefix**.

In [16]:
# Cell 16 — Prefix tuning: reparameterised per-layer key/value prefix, initialised from real words
class Prefix(nn.Module):
    # prefix = P0 + MLP(E), reshaped to L x (K, V) [n_kv_head, P, head_dim]. MLP last layer = 0 => prefix = P0 at init.
    def __init__(self, P0, n_layer, n_kv, hd, d_model, hidden):
        super().__init__()
        self.L, self.n_kv, self.hd, self.P = n_layer, n_kv, hd, P0.shape[0]
        self.register_buffer("P0", P0)                                          # [P, L*2*n_kv*hd] (frozen, real-word KV)
        self.E = nn.Parameter(torch.randn(self.P, d_model) * 0.02)
        self.mlp = nn.Sequential(nn.Linear(d_model, hidden), nn.Tanh(), nn.Linear(hidden, P0.shape[1]))
        nn.init.zeros_(self.mlp[2].weight); nn.init.zeros_(self.mlp[2].bias)

    def forward(self):
        z = (self.P0 + self.mlp(self.E)).view(self.P, self.L, 2, self.n_kv, self.hd)
        z = z.permute(1, 2, 3, 0, 4)                                            # [L, 2, n_kv, P, hd]
        return [(z[l, 0], z[l, 1]) for l in range(self.L)]

def _text_kv(text, P):
    # KV cache of the first P tokens of `text` (repeated if short), computed by the frozen Phase-1 model.
    ids = tokenizer.encode(text)
    while len(ids) < P: ids = ids + ids
    with torch.no_grad():
        _, caches = lm.backbone(torch.tensor([ids[:P]], device=device))
    flat = torch.stack([torch.stack([k[0], v[0]]) for k, v in caches])          # [L, 2, n_kv, P, hd]
    return flat.permute(3, 0, 1, 2, 4).reshape(P, -1).float().cpu(), ids[:P]

def prefix_kv(model, B):
    if getattr(model, "prefix", None) is None: return None
    dt = AMP_DTYPE if (USE_AMP and torch.is_autocast_enabled()) else torch.float32
    return [(k.unsqueeze(0).expand(B, -1, -1, -1).to(dt), v.unsqueeze(0).expand(B, -1, -1, -1).to(dt)) for k, v in model.prefix()]

def build_model():
    # Fresh copy of the frozen Phase-1 model + a trainable prefix.
    hp = HP[METHOD]
    model = copy.deepcopy(lm).cpu()
    for p in model.parameters(): p.requires_grad = False
    P0, init_ids = _text_kv(hp["init_text"], hp["prefix_len"])
    model.prefix = Prefix(P0, lm_cfg.n_layer, lm_cfg.n_kv_head, lm_cfg.head_dim, lm_cfg.d_model, hp["mlp_hidden"])
    model.to(device).eval()
    # check: at init, prefix + x == the Phase-1 model on (init text + x)
    _x = torch.tensor([[BOS_ID] + tokenizer.encode("The appeal is allowed and the judgment of the High Court is set aside.")], device=device)
    with torch.no_grad():
        a, _ = model.backbone(_x, kv_caches=prefix_kv(model, 1))
        b, _ = lm.backbone(torch.cat([torch.tensor([init_ids], device=device), _x], 1))
    err = (a - b[:, len(init_ids):]).abs().max().item()
    print(f"prefix P={hp['prefix_len']} initialised from {repr(tokenizer.decode(init_ids)[:80])}…")
    print(f"init check: max|prefix-model(x) - Phase-1(init text + x)| = {err:.2e}")
    assert err < 1e-3, "the prefix does not reproduce the real-word KV cache at init"
    tot = sum(p.numel() for p in model.parameters()); tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"params: total {tot/1e6:.2f}M | trainable {tr/1e6:.3f}M ({100*tr/tot:.2f}%) | deployed prefix "
          f"{lm_cfg.n_layer * 2 * hp['prefix_len'] * lm_cfg.n_kv_head * lm_cfg.head_dim / 1e6:.2f}M numbers")
    model.n_total, model.n_trainable = tot, tr
    return model

def backbone_hidden(model, ids, mask=None, pos=None):
    # hidden states with the prefix; right padding + causal attention -> no padding mask needed
    h, _ = model.backbone(ids, kv_caches=prefix_kv(model, ids.shape[0]))
    return h

In [17]:
# Cell 17 — Training pairs, micro-batches, loss
def make_pairs(docs, seed, empty_keep=EMPTY_KEEP):
    # -> list of (doc index, chunk index). Every chunk with a target + a random share of the empty ones (EOS-only target).
    rng = random.Random(seed); pairs = []
    for di, d in enumerate(docs):
        for j, t in enumerate(d["tgts"]):
            if t is not None or rng.random() < empty_keep: pairs.append((di, j))
    return pairs

def seq_of(docs, pair):
    di, j = pair; d = docs[di]
    p = prompt_ids(d, j)
    t = d["tgts"][j]
    tgt = ([] if t is None else t.astype(np.int64).tolist()) + ([] if (t is not None and d["trunc"][j]) else [EOS_ID])
    return p, tgt

def make_microbatches(docs, pairs, shuffle, seed=0, budget=None):
    budget = budget or MICRO_TOKENS
    lens = [len(prompt_ids(docs[di], j)) + (0 if docs[di]["tgts"][j] is None else len(docs[di]["tgts"][j])) + 1 for di, j in pairs]
    idx = list(range(len(pairs))); rng = random.Random(seed)
    if shuffle: rng.shuffle(idx)
    batches, pool = [], 2000
    for p in range(0, len(idx), pool):
        part = sorted(idx[p:p + pool], key=lambda i: lens[i])
        cur, longest = [], 0
        for i in part:
            if cur and (len(cur) + 1) * max(longest, lens[i]) > budget:
                batches.append(cur); cur, longest = [], 0
            cur.append(i); longest = max(longest, lens[i])
        if cur: batches.append(cur)
    if shuffle: rng.shuffle(batches)
    return [[pairs[i] for i in b] for b in batches]

def build_batch(docs, pairs):
    seqs, labs = [], []
    for pr in pairs:
        p, t = seq_of(docs, pr)
        full = p + t
        y = [-100] * (len(p) - 1) + t
        seqs.append(full[:-1]); labs.append(y)
    T = max(len(s) for s in seqs)
    x = np.full((len(seqs), T), PAD_ID, dtype=np.int64); y = np.full((len(seqs), T), -100, dtype=np.int64)
    for i, (s, l) in enumerate(zip(seqs, labs)):
        x[i, :len(s)] = s; y[i, :len(l)] = l
    return torch.from_numpy(x).to(device), torch.from_numpy(y).to(device)

def batch_loss(model, x, y):
    # sum of target-token NLL and the number of target tokens. No padding mask needed (right padding + causal attention).
    with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
        h = backbone_hidden(model, x)
        sel = y != -100
        logits = model.lm_head(h[sel])
    return F.cross_entropy(logits.float(), y[sel], reduction="sum"), int(sel.sum().item())

@torch.no_grad()
def eval_loss(model, docs, pairs):
    model.eval(); tot, n = 0.0, 0
    for b in make_microbatches(docs, pairs, shuffle=False, budget=2 * MICRO_TOKENS):
        x, y = build_batch(docs, b); l, k = batch_loss(model, x, y); tot += l.item(); n += k
    return tot / max(n, 1)

In [18]:
# Cell 18 — Batched greedy generation with KV cache, document summarisation, ROUGE
@torch.no_grad()
def generate(model, prompts, max_new, min_new=None, no_repeat_ngram=0, rep_penalty=1.0, amp=True, eos_threshold=0.0):
    # prompts: list of token-id lists (left-padded here). The prefix (if any) is the initial KV cache. Returns ids without EOS.
    model.eval(); B = len(prompts); L = max(len(p) for p in prompts)
    ids = torch.full((B, L), PAD_ID, dtype=torch.long, device=device); mask = torch.zeros((B, L), dtype=torch.long, device=device)
    for i, p in enumerate(prompts):
        ids[i, L - len(p):] = torch.tensor(p, device=device); mask[i, L - len(p):] = 1
    min_new_t = torch.tensor(min_new if min_new is not None else [0] * B, device=device)
    with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP and amp):
        pk = prefix_kv(model, B)
        if pk is not None: mask = torch.cat([torch.ones((B, pk[0][0].shape[2]), dtype=mask.dtype, device=device), mask], 1)
        h, caches = model.backbone(ids, attention_mask=mask, kv_caches=pk)
        logits = model.lm_head(h[:, -1]).float()
    out = [[] for _ in range(B)]; done = [False] * B; bans = [defaultdict(set) for _ in range(B)]
    seen = torch.zeros((B, logits.shape[-1]), dtype=torch.bool, device=device); n = no_repeat_ngram
    for t in range(max_new):
        lg = logits
        if rep_penalty != 1.0:
            lg = torch.where(seen, torch.where(lg > 0, lg / rep_penalty, lg * rep_penalty), lg)
        lg[:, PAD_ID] = float("-inf"); lg[:, BOS_ID] = float("-inf")
        if eos_threshold > 0:
            lg[:, EOS_ID] = torch.where(torch.softmax(lg, -1)[:, EOS_ID] < eos_threshold, torch.full_like(lg[:, EOS_ID], float("-inf")), lg[:, EOS_ID])
        lg[:, EOS_ID] = torch.where(t < min_new_t, torch.full_like(lg[:, EOS_ID], float("-inf")), lg[:, EOS_ID])
        if n > 0 and t >= n - 1:
            rows, cols = [], []
            for b in range(B):
                if done[b]: continue
                for tok in bans[b].get(tuple(out[b][-(n - 1):]) if n > 1 else (), ()): rows.append(b); cols.append(tok)
            if rows: lg[rows, cols] = float("-inf")
        nl = lg.argmax(-1).tolist()
        for b in range(B):
            if done[b]: nl[b] = PAD_ID; continue
            if nl[b] == EOS_ID: done[b] = True; continue
            out[b].append(nl[b])
            if n > 0 and len(out[b]) >= n: bans[b][tuple(out[b][-n:-1]) if n > 1 else ()].add(out[b][-1])
        if all(done): break
        nxt = torch.tensor(nl, device=device)
        seen[torch.arange(B, device=device), nxt] = True
        mask = torch.cat([mask, torch.ones((B, 1), dtype=mask.dtype, device=device)], 1)
        with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP and amp):
            h, caches = model.backbone(nxt[:, None], attention_mask=mask, kv_caches=caches)
            logits = model.lm_head(h[:, -1]).float()
    return out

def select_chunks(n, k):
    # Head + tail + evenly spread middle (the audit's U-shaped source positions).
    if n <= k: return list(range(n))
    h = max(1, k // 4); t = max(1, k // 4); m = k - h - t
    mid = np.linspace(h, n - t - 1, m).round().astype(int).tolist() if m > 0 else []
    return sorted(set(list(range(h)) + mid + list(range(n - t, n))))

def predict_len_words(doc_words):
    return float(np.clip(LEN_MODEL["a"] * max(doc_words, 1) ** LEN_MODEL["b"], LEN_MODEL["lo"], LEN_MODEL["hi"]))

def _dedup(text):
    seen, out = set(), []
    for _, s in split_sentences(text):
        k = " ".join(words(s))
        if k and k in seen: continue
        seen.add(k); out.append(s)
    return " ".join(out)

def _dedup_parts(segs):
    # sentence-level de-duplication across the chunk outputs (keeps chunk boundaries)
    seen, out = set(), []
    for t in segs:
        keep = []
        for _, s in split_sentences(t):
            k = " ".join(words(s))
            if k and k in seen: continue
            seen.add(k); keep.append(s)
        if keep: out.append(" ".join(keep))
    return out

def _cut_words(text, n):
    # first n words, ending at the last sentence end in the kept part if that keeps >= 60 % of n
    w = text.split()
    if len(w) <= n: return text
    head = " ".join(w[:n]); m = max(head.rfind(". "), head.rfind("? "), head.rfind("! "))
    if head.endswith((".", "?", "!")): return head
    return head[:m + 1] if m >= 0 and len(head[:m + 1].split()) >= 0.6 * n else head

def _trim(segs, budget, mode):
    tot = sum(len(t.split()) for t in segs)
    if tot <= budget: return " ".join(segs)
    if mode == "tail": return " ".join(" ".join(segs).split()[:budget])
    out = [_cut_words(t, max(1, int(len(t.split()) * budget / tot))) for t in segs]     # every chunk keeps the same share
    return " ".join(x for x in out if x)

LAST_GEN_STATS = {}
def summarize_docs(model, docs, gen=None, return_parts=False):
    g = dict(GEN, **(gen or {}))
    jobs = []                                                               # (doc index, chunk index)
    for di, d in enumerate(docs):
        for j in select_chunks(len(d["chunks"]), g["max_chunks"]): jobs.append((di, j))
    prompts = [prompt_ids(docs[di], j) for di, j in jobs]
    first = {di: min(j for dj, j in jobs if dj == di) for di in range(len(docs))}
    order = sorted(range(len(jobs)), key=lambda i: len(prompts[i]))
    outs = [None] * len(jobs); t0 = time.time()
    for s in range(0, len(order), g["batch"]):
        bi = order[s:s + g["batch"]]
        mins = [max(g["min_new_all"], g["min_new_first"] if jobs[i][1] == first[jobs[i][0]] else 0) for i in bi]
        res = generate(model, [prompts[i] for i in bi], g["max_new"], mins, g["no_repeat_ngram"], g["rep_penalty"], eos_threshold=g["eos_threshold"])
        for i, r in zip(bi, res): outs[i] = r
    parts = defaultdict(list)
    for (di, j), o in zip(jobs, outs): parts[di].append((j, tokenizer.decode(o).strip()))
    LAST_GEN_STATS.update(chunks=len(jobs), empty_share=float(np.mean([len(o) == 0 for o in outs])),
                          tokens_per_chunk=float(np.mean([len(o) for o in outs])), hit_max_share=float(np.mean([len(o) >= g["max_new"] for o in outs])))
    summaries = []
    for di, d in enumerate(docs):
        segs = [t for _, t in sorted(parts[di]) if t]
        if g["dedup"]: segs = _dedup_parts(segs)
        budget = int(round(g["len_factor"] * predict_len_words(d["words"])))
        summaries.append(_trim(segs, budget, g["trim"]))
    secs = time.time() - t0
    return (summaries, secs, parts) if return_parts else (summaries, secs)

# ---- ROUGE: fast exact implementation verified against rouge_score (as in the audit, Cell 9) ----
try:
    from rouge_score import rouge_scorer
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rouge-score"], check=True)
    from rouge_score import rouge_scorer
_RS = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
def rtok(text): return _RS._tokenizer.tokenize(text)
def lcs_len(a, b):
    if not a or not b: return 0
    masks = {}
    for i, x in enumerate(a): masks[x] = masks.get(x, 0) | (1 << i)
    full = (1 << len(a)) - 1; V = full
    for y in b:
        U = V & masks.get(y, 0)
        V = ((V + U) | (V - U)) & full
    return len(a) - bin(V).count("1")
def _f(o, nc, nr):
    p, r = (o / nc if nc else 0.0), (o / nr if nr else 0.0)
    return p, r, (2 * p * r / (p + r) if p + r else 0.0)
def rouge(cand, ref):
    c, r = rtok(cand), rtok(ref); out = {}
    for n in (1, 2):
        cc, rc = Counter(zip(*[c[i:] for i in range(n)])), Counter(zip(*[r[i:] for i in range(n)]))
        out[f"r{n}"] = _f(sum(min(v, rc[k]) for k, v in cc.items()), sum(cc.values()), sum(rc.values()))
    out["rL"] = _f(lcs_len(r, c), len(c), len(r))
    return out
_c, _r = "the appellant was convicted under section 302 by the court", "the appellant was convicted by the sessions court under section 302"
assert all(abs(_RS.score(_r, _c)[k].fmeasure - rouge(_c, _r)[k2][2]) < 1e-9 for k, k2 in (("rouge1", "r1"), ("rouge2", "r2"), ("rougeL", "rL")))

def rouge_eval(cands, refs):
    R = defaultdict(list)
    for c, r in zip(cands, refs):
        for k, v in rouge(c, r).items(): R[k].append(v)
    return {k: dict(p=float(np.mean([x[0] for x in v])), r=float(np.mean([x[1] for x in v])), f=float(np.mean([x[2] for x in v])))
            for k, v in R.items()}, {k: np.asarray([x[2] for x in v]) for k, v in R.items()}

In [19]:
# Cell 19 — Self-test: batched + cached + left-padded greedy == one-sequence, no-cache greedy, WITH the prefix
def _greedy_ref(model, p, steps):
    seq = list(p); out = []
    for _ in range(steps):
        with torch.no_grad():
            h, _ = model.backbone(torch.tensor([seq], device=device), kv_caches=prefix_kv(model, 1))
            lg = model.lm_head(h[:, -1]).float()[0]
        lg[PAD_ID] = float("-inf"); lg[BOS_ID] = float("-inf")
        t = int(lg.argmax())
        if t == EOS_ID: break
        out.append(t); seq.append(t)
    return out

_tf32 = torch.backends.cuda.matmul.allow_tf32
torch.backends.cuda.matmul.allow_tf32 = False
_tm = build_model()
with torch.no_grad():                                                    # perturb the prefix so the test is not trivially the init
    _tm.prefix.mlp[2].weight.normal_(0, 0.02)
_tp = [[BOS_ID] + tokenizer.encode("The appellant was convicted by the Sessions Court and"), [BOS_ID] + tokenizer.encode("Held:"),
       [BOS_ID] + tokenizer.encode(VAL_ALL[0]["doc"][:2400])[:420]]
_steps = 24
for _m, _name in ((lm, "Phase-1 (no prefix)"), (_tm, "with a prefix")):
    _b = generate(_m, _tp, _steps, amp=False)
    for p, g in zip(_tp, _b):
        r = _greedy_ref(_m, p, _steps)
        assert g == r, f"[{_name}] cached batched generation differs from the reference:\n{g}\n{r}"
    print(f"generation self-test OK [{_name}]: batched + KV cache + left padding == reference greedy (prompts {[len(p) for p in _tp]})")
torch.backends.cuda.matmul.allow_tf32 = _tf32
del _tm; gc.collect()
print("Phase-1 continuation (no fine-tuning):", repr(tokenizer.decode(generate(lm, _tp[:1], 40)[0])[:200]))

prefix P=64 initialised from 'The following is a summary of a judgment of the Supreme Court of India. It state'…
init check: max|prefix-model(x) - Phase-1(init text + x)| = 9.78e-06
params: total 56.62M | trainable 1.684M (2.97%) | deployed prefix 0.79M numbers
generation self-test OK [Phase-1 (no prefix)]: batched + KV cache + left padding == reference greedy (prompts [10, 4, 421])
generation self-test OK [with a prefix]: batched + KV cache + left padding == reference greedy (prompts [10, 4, 421])
Phase-1 continuation (no fine-tuning): ' sentenced to undergo rigorous imprisonment for life.\n\nThe appellant was sentenced to undergo rigorous imprisonment for life.\n\nThe appellant was sentenced to undergo rigorous imprisonment for life.\n\nT'


## Build the data and train

In [20]:
# Cell 20 — Prepare data, build the model, train
seed_everything(SEED)
if SMOKE_TEST:
    _r = random.Random(SEED)
    train_docs = _r.sample(TR_ALL, min(SMOKE_TRAIN_DOCS, len(TR_ALL))); val_docs = VAL_ALL[:SMOKE_VAL_DOCS]
else:
    train_docs, val_docs = TR_ALL, VAL_ALL
_t0 = time.time()
if RESUME_FROM is None or EXPORT_TEACHER:                                   # decode-only runs need the TR chunks only for the export
    prepare_docs(train_docs, "tr_smoke" if SMOKE_TEST else "tr", True); pair_stats(train_docs, "TR")
prepare_docs(val_docs, "val_smoke" if SMOKE_TEST else "val", True)
print(f"preparation {time.time() - _t0:.0f}s")
pair_stats(val_docs, "VAL")
val_pairs = make_pairs(val_docs, seed=0, empty_keep=EMPTY_KEEP)

def param_groups(model):
    hp = HP[METHOD]
    ps = [p for p in model.parameters() if p.requires_grad]
    return [dict(params=ps, lr=hp["lr"], base_lr=hp["lr"], weight_decay=hp["wd"])]

def quick_rouge(model, docs, n):
    if not n: return None
    s, _ = summarize_docs(model, docs[:n])
    m, _ = rouge_eval(s, [d["summ"] for d in docs[:n]])
    return m

def train_model(model, epochs, max_steps=None):
    opt = torch.optim.AdamW(param_groups(model), betas=(0.9, 0.999))
    scaler = torch.amp.GradScaler("cuda", enabled=USE_SCALER)
    trainables = [p for p in model.parameters() if p.requires_grad]
    n_pairs = len(make_pairs(train_docs, seed=SEED + 1))
    steps_per_epoch = math.ceil(n_pairs / STEP_SEQS); total = epochs * steps_per_epoch
    if max_steps: total = min(total, max_steps)
    warm = max(1, int(WARMUP_FRAC * total))
    def lr_mult(s):
        if s < warm: return (s + 1) / warm
        return 0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * min(1.0, (s - warm) / max(1, total - warm))))
    print(f"~{n_pairs} training sequences/epoch | ~{steps_per_epoch} steps/epoch ({STEP_SEQS} seqs/step) | total {total} steps")
    if device.type == "cuda": torch.cuda.reset_peak_memory_stats()
    history, losses, step, done = [], [], 0, False
    best = dict(score=None, epoch=0, state=None); t_train = time.time(); tok_seen = 0
    v0 = eval_loss(model, val_docs, val_pairs); print(f"VAL loss before training (Phase-1 model, target tokens): {v0:.4f}")
    for ep in range(1, epochs + 1):
        pairs = make_pairs(train_docs, seed=SEED + ep)
        batches = make_microbatches(train_docs, pairs, shuffle=True, seed=SEED + ep)
        model.train(); t0 = time.time(); ep_sum, ep_tok, acc, acc_tok, acc_loss = 0.0, 0, 0, 0, None
        for bi, b in enumerate(batches):
            x, y = build_batch(train_docs, b)
            l, k = batch_loss(model, x, y)
            # normalise by the target tokens of the whole step: sum-losses are divided by an estimate (k * STEP_SEQS / len(b))
            scaler.scale(l / max(1.0, k * STEP_SEQS / len(b))).backward()
            ep_sum += l.item(); ep_tok += k; tok_seen += x.numel(); acc += len(b)
            losses.append(l.item() / max(k, 1))
            if acc >= STEP_SEQS or bi == len(batches) - 1:
                m = lr_mult(step)
                for g in opt.param_groups: g["lr"] = g["base_lr"] * m
                scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(trainables, GRAD_CLIP)
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); step += 1; acc = 0
                if step == 20:
                    per = (time.time() - t0) / 20
                    print(f"  >> speed: {per:.2f} s/step | projected {per * steps_per_epoch / 60:.1f} min/epoch "
                          f"({per * steps_per_epoch * epochs / 60:.0f} min for {epochs} epochs, plus evaluation)"
                          + (f" | peak GPU {torch.cuda.max_memory_allocated() / 2**30:.1f} GB" if device.type == "cuda" else ""))
                if step % 100 == 0:
                    print(f"  ep {ep} step {step}/{total} | loss {np.mean(losses[-300:]):.4f} | lr x{m:.3f} | {time.time() - t0:.0f}s")
                if max_steps and step >= max_steps: done = True; break
                if TIME_BUDGET_MIN and (time.time() - t_train) / 60 > TIME_BUDGET_MIN:
                    print(f"  TIME_BUDGET_MIN={TIME_BUDGET_MIN} reached -> stopping"); done = True; break
        vl = eval_loss(model, val_docs, val_pairs)
        qr = quick_rouge(model, val_docs, min(VAL_GEN_DOCS_EPOCH, len(val_docs)))
        rec = dict(epoch=ep, train_loss=ep_sum / max(ep_tok, 1), val_loss=vl, seconds=time.time() - t0, steps=step)
        if qr: rec.update(q_r1=qr["r1"]["f"], q_r2=qr["r2"]["f"], q_rL=qr["rL"]["f"])
        score = -vl if SELECT_BY == "loss" else (qr["r1"]["f"] + qr["rL"]["f"]) / 2
        if best["score"] is None or score > best["score"]:
            best = dict(score=score, epoch=ep, state={n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad})
            torch.save(best, RUN_DIR / "best_trainable.pt")
        print(f"epoch {ep}: " + " | ".join(f"{k} {v:.4f}" if isinstance(v, float) else f"{k} {v}" for k, v in rec.items() if k != "epoch"))
        history.append(rec)
        if done: break
    stats = dict(train_seconds=time.time() - t_train, steps=step, tokens_seen=tok_seen, val_loss_init=v0,
                 peak_gpu_gb=(torch.cuda.max_memory_allocated() / 2**30) if device.type == "cuda" else 0.0,
                 first_losses=float(np.mean(losses[:20])) if losses else None, last_losses=float(np.mean(losses[-20:])) if losses else None)
    return history, best, stats

model = build_model()
if RESUME_FROM is None:
    history, best, stats = train_model(model, EPOCHS, max_steps=SMOKE_STEPS if SMOKE_TEST else None)
    model.load_state_dict(best["state"], strict=False)
    print(f"\nbest epoch {best['epoch']} (by {SELECT_BY}) | train {stats['train_seconds']/60:.1f} min | peak GPU {stats['peak_gpu_gb']:.1f} GB")
else:
    _r = torch.load(RESUME_FROM, map_location="cpu", weights_only=False)
    _st = _r.get("trainable") or _r.get("state")
    _miss = [n for n, p in model.named_parameters() if p.requires_grad and n not in _st]
    assert not _miss, f"{RESUME_FROM} lacks trainable weights: {_miss[:3]}"
    model.load_state_dict(_st, strict=False)
    _log = _r.get("log", {})
    history, best = _log.get("history", []), dict(epoch=_log.get("best_epoch", 0), state=_st)
    stats = dict(_log.get("stats", {}), resumed_from=str(RESUME_FROM))
    print(f"DECODE-ONLY: loaded {len(_st)} trainable tensors from {RESUME_FROM} (best epoch {best['epoch']})")
    if val_docs: print(f"VAL loss of the loaded weights: {eval_loss(model, val_docs, val_pairs):.4f}")

prepare tr:   0%|          | 0/6759 [00:00<?, ?it/s]

TR: 6759 docs | chunks/doc mean 10.2 median 8 | chunk pairs with a target 47129 | empty chunks 31.8% | target tokens median 116 p90 277 | single-sentence cut 0.1%


prepare val:   0%|          | 0/200 [00:00<?, ?it/s]

preparation 132s
VAL: 200 docs | chunks/doc mean 11.0 median 9 | chunk pairs with a target 1522 | empty chunks 30.8% | target tokens median 110 p90 280 | single-sentence cut 0.1%
prefix P=64 initialised from 'The following is a summary of a judgment of the Supreme Court of India. It state'…
init check: max|prefix-model(x) - Phase-1(init text + x)| = 9.78e-06
params: total 56.62M | trainable 1.684M (2.97%) | deployed prefix 0.79M numbers
~58256 training sequences/epoch | ~1821 steps/epoch (32 seqs/step) | total 7284 steps
VAL loss before training (Phase-1 model, target tokens): 2.6188
  >> speed: 0.74 s/step | projected 22.6 min/epoch (90 min for 4 epochs, plus evaluation) | peak GPU 4.8 GB
  ep 1 step 100/7284 | loss 3.5646 | lr x0.459 | 73s
  ep 1 step 200/7284 | loss 2.2864 | lr x0.917 | 146s
  ep 1 step 300/7284 | loss 2.1051 | lr x1.000 | 219s
  ep 1 step 400/7284 | loss 1.9951 | lr x0.999 | 295s
  ep 1 step 500/7284 | loss 1.9343 | lr x0.996 | 367s
  ep 1 step 600/7284 | loss 1.90

In [21]:
# Cell 21 — Smoke-test verdict + sample outputs (always printed)
val_extra = {}
_n = min(2, len(val_docs))
_s, _, _parts = summarize_docs(model, val_docs[:_n], return_parts=True)
for d, s in zip(val_docs[:_n], _s):
    print(f"--- {d['id']} | {len(d['chunks'])} chunks | generated {len(s.split())} words | reference {len(d['summ'].split())} words")
    print("GENERATED:", s[:700].replace("\n", " "))
    print("REFERENCE:", d["summ"][:700].replace("\n", " "))
if SMOKE_TEST:
    drop = stats["last_losses"] / stats["first_losses"]
    _ss = summarize_docs(model, val_docs)[0]
    m, _ = rouge_eval(_ss, [d["summ"] for d in val_docs])
    val_extra.update(smoke_r1=m["r1"]["f"], smoke_r1_p=m["r1"]["p"], smoke_r1_r=m["r1"]["r"], smoke_rL=m["rL"]["f"],
                     smoke_words_gen=float(np.mean([len(x.split()) for x in _ss])), smoke_words_ref=float(np.mean([len(d["summ"].split()) for d in val_docs])),
                     smoke_gen_stats=dict(LAST_GEN_STATS))
    print(f"generated {val_extra['smoke_words_gen']:.0f} words vs reference {val_extra['smoke_words_ref']:.0f} | R1 P {m['r1']['p']:.3f} R {m['r1']['r']:.3f} | "
          f"chunks with an empty output {LAST_GEN_STATS['empty_share']:.2f} | tokens/chunk {LAST_GEN_STATS['tokens_per_chunk']:.0f}")
    print(f"\nloss {stats['first_losses']:.3f} -> {stats['last_losses']:.3f} (x{drop:.2f}) | VAL loss {history[-1]['val_loss']:.3f} "
          f"(Phase-1 {stats['val_loss_init']:.3f}) | R1 {m['r1']['f']:.3f} RL {m['rL']['f']:.3f} on {len(val_docs)} docs")
    print("SMOKE VERDICT:", "OK -> run the full job" if (drop < 0.95 and history[-1]["val_loss"] < stats["val_loss_init"]) else
          "NOT OK: loss did not fall / VAL loss did not improve -> paste the log to Claude")
    print("Check the '>> speed' line: it projects minutes per epoch; set EPOCHS / TIME_BUDGET_MIN accordingly.")

--- 2459 | 12 chunks | generated 907 words | reference 912 words
GENERATED: The first respondent was one of the candidates who had contested the election and filed his nomination paper on June 24, 1967. The first respondent was a candidate who had contested it in the High Court. The first petitioner was a candidate belonging to the second respondent. The fourth respondent was a person who had contested that election. The election petitioner filed a petition for withdrawing the allegations against V.K. Reddi, and the second respondent who was guilty of such a practice by making payment of illegal gratification. The trial Judge held that the allegations contained in the election petition were not liable to be rejected. The High Court dismissed the application. The Hig
REFERENCE: The election of the first respondent to the Andhra Pradesh Legislative Assembly at the General Election hold in 1967 was challenged by the appellants in an election petition. Various corrupt practices were allege

## Decoding sweep and VAL evaluation (full runs)
The decoding configuration is chosen on the first `SWEEP_DOCS` VAL documents by (R1 + RL)/2, then **all 200** VAL documents are summarised with it (the audit's baselines were scored on the same 200). The analysis covers what the PDF asks for: scores vs the extractive baselines, length vs score, redundancy (repeated n-grams), abstractiveness of the outputs (novel n-grams vs the document), scores by document length, and cost.

In [22]:
# Cell 22 — Decoding sweep (on VAL[:SWEEP_DOCS]) -> best GEN
_BS = None
def bertscore_f(cands, refs):
    # roberta-large BERTScore (the competition metric), loaded once; None without a GPU / when disabled
    global _BS
    if not (RUN_BERTSCORE and device.type == "cuda"): return None
    if _BS is None:
        try:
            import bert_score
        except ImportError:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bert-score"], check=True); import bert_score
        import transformers; transformers.logging.set_verbosity_error()
        _BS = bert_score.BERTScorer(model_type=BERTSCORE_MODEL, lang="en", device="cuda", batch_size=8)
    P_, R_, F_ = _BS.score(cands, refs)
    return P_.numpy(), R_.numpy(), F_.numpy()

sweep_res = []
if not SMOKE_TEST and SWEEP_DOCS and DECODE_SWEEP:
    _docs = val_docs[:SWEEP_DOCS]; _refs = [d["summ"] for d in _docs]
    for cfg in DECODE_SWEEP:
        s, secs = summarize_docs(model, _docs, cfg); m, _ = rouge_eval(s, _refs)
        wg = float(np.mean([len(x.split()) for x in s])); bs = bertscore_f(s, _refs)
        r = dict(cfg=cfg, r1=m["r1"]["f"], r2=m["r2"]["f"], rL=m["rL"]["f"], r1_p=m["r1"]["p"], r1_r=m["r1"]["r"], words=wg,
                 empty_chunks=LAST_GEN_STATS["empty_share"], tok_chunk=LAST_GEN_STATS["tokens_per_chunk"], secs=secs)
        if bs is not None: r["bs_f"] = float(bs[2].mean())
        sweep_res.append(r)
        print(f"  {cfg}: R1 {r['r1']:.4f} (P {r['r1_p']:.3f} R {r['r1_r']:.3f}) R2 {r['r2']:.4f} RL {r['rL']:.4f}"
              + (f" | BS {r['bs_f']:.4f}" if "bs_f" in r else "") + f" | words {wg:.0f} | tok/chunk {r['tok_chunk']:.0f} | "
              f"empty {r['empty_chunks']:.2f} | {secs:.0f}s")
    use_bs = SWEEP_SELECT == "bertscore" and all("bs_f" in r for r in sweep_res)
    bestcfg = max(sweep_res, key=lambda r: r["bs_f"] if use_bs else (r["r1"] + r["rL"]) / 2)["cfg"]
    GEN.update(bestcfg); print(f"chosen decoding (by {'BERTScore' if use_bs else '(R1 + RL) / 2'}):", bestcfg)
else:
    print("decoding sweep skipped")

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.42GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

  {'min_new_all': 64}: R1 0.5180 (P 0.613 R 0.499) R2 0.2451 RL 0.2514 | BS 0.8409 | words 666 | tok/chunk 84 | empty 0.00 | 66s
  {'min_new_all': 96}: R1 0.5304 (P 0.569 R 0.549) R2 0.2541 RL 0.2536 | BS 0.8401 | words 777 | tok/chunk 116 | empty 0.00 | 87s
  {'min_new_all': 128}: R1 0.5313 (P 0.548 R 0.569) R2 0.2546 RL 0.2530 | BS 0.8402 | words 836 | tok/chunk 150 | empty 0.00 | 98s
  {'min_new_all': 32, 'eos_threshold': 0.5}: R1 0.5256 (P 0.571 R 0.541) R2 0.2503 RL 0.2490 | BS 0.8389 | words 785 | tok/chunk 122 | empty 0.00 | 139s
  {'min_new_all': 64, 'eos_threshold': 0.5}: R1 0.5327 (P 0.556 R 0.563) R2 0.2537 RL 0.2497 | BS 0.8389 | words 827 | tok/chunk 148 | empty 0.00 | 139s
  {'min_new_all': 96, 'trim': 'tail'}: R1 0.5301 (P 0.552 R 0.570) R2 0.2554 RL 0.2514 | BS 0.8401 | words 854 | tok/chunk 116 | empty 0.00 | 87s
  {'min_new_all': 96, 'max_chunks': 24}: R1 0.5318 (P 0.566 R 0.554) R2 0.2551 RL 0.2537 | BS 0.8398 | words 801 | tok/chunk 116 | empty 0.00 | 89s
  {'min_ne

In [23]:
# Cell 23 — VAL evaluation: ROUGE (+ BERTScore), lengths, redundancy, abstractiveness, by document length
val_res = {}
if not SMOKE_TEST:
    refs = [d["summ"] for d in val_docs]
    val_sum, t_val = summarize_docs(model, val_docs)
    val_extra["gen_stats"] = dict(LAST_GEN_STATS)
    m, per = rouge_eval(val_sum, refs)
    val_res = {k: v["f"] for k, v in m.items()}
    val_extra.update(seconds=t_val, sec_per_doc=t_val / len(val_docs),
                     words_gen=float(np.mean([len(s.split()) for s in val_sum])), words_ref=float(np.mean([len(r.split()) for r in refs])),
                     r1_p=m["r1"]["p"], r1_r=m["r1"]["r"])
    _b = bertscore_f(val_sum, refs)
    if _b is not None:
        val_res.update(bs_p=float(_b[0].mean()), bs_r=float(_b[1].mean()), bs_f=float(_b[2].mean())); per["bs"] = _b[2]
    print(f"===== VAL ({len(val_docs)} docs) | R1 {val_res['r1']:.4f} (P {m['r1']['p']:.3f} R {m['r1']['r']:.3f}) | R2 {val_res['r2']:.4f} | "
          f"RL {val_res['rL']:.4f}" + (f" | BERTScore F1 {val_res['bs_f']:.4f}" if "bs_f" in val_res else ""))
    for name, b in BASELINES.items():
        print(f"      vs {name:6s}: R1 {val_res['r1'] - b['r1']:+.4f} | R2 {val_res['r2'] - b['r2']:+.4f} | RL {val_res['rL'] - b['rL']:+.4f}"
              + (f" | BERTScore {val_res['bs_f'] - b['bs']:+.4f}" if "bs_f" in val_res else ""))
    print(f"      words: generated {val_extra['words_gen']:.0f} vs reference {val_extra['words_ref']:.0f} | {val_extra['sec_per_doc']:.1f} s/doc")

    # redundancy: share of repeated 3-grams inside each output; abstractiveness: novel n-grams w.r.t. the document
    def rep3(t):
        w = words(t); g = list(zip(w, w[1:], w[2:]))
        return 1 - len(set(g)) / max(len(g), 1)
    def novel(t, d, n):
        a, s = words(d), words(t); ga = set(zip(*[a[i:] for i in range(n)])); gs = list(zip(*[s[i:] for i in range(n)]))
        return sum(x not in ga for x in gs) / max(len(gs), 1)
    val_extra["rep3_gen"] = float(np.mean([rep3(s) for s in val_sum])); val_extra["rep3_ref"] = float(np.mean([rep3(r) for r in refs]))
    val_extra["novel"] = {n: float(np.mean([novel(s, d["doc"], n) for s, d in zip(val_sum, val_docs)])) for n in (1, 2, 4)}
    print(f"      redundancy (repeated 3-grams): generated {val_extra['rep3_gen']:.3f} vs reference {val_extra['rep3_ref']:.3f}")
    print(f"      novel n-grams vs the document (1/2/4): {[round(v, 3) for v in val_extra['novel'].values()]}  (references: 0.044 / 0.223 / 0.484)")

    ntok_v = np.asarray([d["ntok"] for d in val_docs]); cuts = np.percentile(ntok_v, [33.3, 66.7]); bk = np.digitize(ntok_v, cuts); rows = []
    print("      by document length (token terciles):")
    for b_, name in enumerate(("short", "medium", "long")):
        sel = bk == b_
        r_ = dict(bucket=name, n=int(sel.sum()), median_tokens=float(np.median(ntok_v[sel])), r1=float(per["r1"][sel].mean()),
                  rL=float(per["rL"][sel].mean()), truncated=float(np.mean([len(val_docs[i]["chunks"]) > GEN["max_chunks"] for i in np.where(sel)[0]])))
        if "bs" in per: r_["bs"] = float(per["bs"][sel].mean())
        rows.append(r_)
        print(f"        {name:6s}: {sel.sum():3d} docs | median {r_['median_tokens']:6.0f} tokens | R1 {r_['r1']:.3f} RL {r_['rL']:.3f}"
              + (f" BS {r_['bs']:.3f}" if "bs" in r_ else "") + f" | chunks skipped in {100 * r_['truncated']:.0f}% of docs")
    val_extra["by_length"] = rows
    (RUN_DIR / "val_summaries.json").write_text(json.dumps([dict(id=d["id"], summary=s) for d, s in zip(val_docs, val_sum)], ensure_ascii=False))
else:
    print("no VAL evaluation (smoke run)")

===== VAL (200 docs) | R1 0.5170 (P 0.609 R 0.499) | R2 0.2455 | RL 0.2519 | BERTScore F1 0.8415
      vs tfidf : R1 -0.0579 | R2 -0.0731 | RL -0.0489 | BERTScore -0.0118
      vs lead  : R1 -0.0052 | R2 -0.0053 | RL +0.0082 | BERTScore +0.0094
      vs oracle: R1 -0.2690 | R2 -0.3587 | RL -0.2822 | BERTScore -0.0440
      words: generated 649 vs reference 848 | 0.6 s/doc
      redundancy (repeated 3-grams): generated 0.159 vs reference 0.099
      novel n-grams vs the document (1/2/4): [0.068, 0.29, 0.65]  (references: 0.044 / 0.223 / 0.484)
      by document length (token terciles):
        short :  67 docs | median   2033 tokens | R1 0.484 RL 0.256 BS 0.846 | chunks skipped in 0% of docs
        medium:  66 docs | median   4260 tokens | R1 0.525 RL 0.258 BS 0.841 | chunks skipped in 0% of docs
        long  :  67 docs | median   8362 tokens | R1 0.542 RL 0.241 BS 0.838 | chunks skipped in 54% of docs


In [24]:
# Cell 24 — Test summaries -> prediction.json (list format) -> submission.zip (full runs only)
def verify_submission(zip_path, docs):
    with zipfile.ZipFile(zip_path) as z:
        names = z.namelist()
        assert names == ["prediction.json"], f"the zip must contain ONLY prediction.json at its root, found {names}"
        sub = json.loads(z.read("prediction.json"))
    assert isinstance(sub, list), "prediction.json must be a JSON list of {id, summary}"
    want = [str(d["id"]) for d in docs]; got = [str(e["id"]) for e in sub]
    assert len(got) == len(set(got)), "duplicate ids"
    assert set(got) == set(want), f"id mismatch: missing {len(set(want) - set(got))}, extra {len(set(got) - set(want))}"
    for e in sub:
        assert set(e) == {"id", "summary"}, f"unexpected keys {set(e)}"
        assert isinstance(e["summary"], str) and e["summary"].strip(), f"{e['id']}: empty summary"
    print(f"verified: {len(sub)} summaries, one non-empty string per test id")

if not SMOKE_TEST:
    prepare_docs(TEST_ALL, "test", False)
    test_sum, t_test = summarize_docs(model, TEST_ALL)
    sub = [dict(id=d["id"], summary=s) for d, s in zip(TEST_ALL, test_sum)]          # id keeps its type from test.json
    out_dir = RUN_DIR / "submission"; out_dir.mkdir(parents=True, exist_ok=True)
    (out_dir / "prediction.json").write_text(json.dumps(sub, ensure_ascii=False))
    with zipfile.ZipFile(out_dir / "submission.zip", "w", zipfile.ZIP_DEFLATED) as z:
        z.write(out_dir / "prediction.json", arcname="prediction.json")
    verify_submission(out_dir / "submission.zip", TEST_ALL)
    print(f"test: {len(sub)} docs in {t_test:.0f}s | words/summary mean {np.mean([len(s.split()) for s in test_sum]):.0f}")
else:
    print("smoke run: no submission written")

prepare test:   0%|          | 0/100 [00:00<?, ?it/s]

verified: 100 summaries, one non-empty string per test id
test: 100 docs in 51s | words/summary mean 630


In [25]:
# Cell 25 — Save trainable weights + run log
def _jsonable(o):
    if isinstance(o, dict): return {str(k): _jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [_jsonable(v) for v in o]
    if isinstance(o, np.ndarray): return o.tolist()
    if isinstance(o, (np.floating, np.integer, np.bool_)): return o.item()
    if isinstance(o, Path): return str(o)
    return o
log = dict(run=RUN_NAME, method=METHOD, hp=HP[METHOD], chunk_tokens=CHUNK_TOKENS, tgt_max=TGT_MAX, empty_keep=EMPTY_KEEP,
           min_align_f1=MIN_ALIGN_F1, epochs=EPOCHS, micro_tokens=MICRO_TOKENS, step_seqs=STEP_SEQS, select_by=SELECT_BY,
           gen=GEN, decode_sweep=sweep_res, params_total=model.n_total, params_trainable=model.n_trainable,
           history=history, best_epoch=best["epoch"], stats=stats, val=val_res, val_extra=val_extra, baselines=BASELINES,
           n_train_docs=len(train_docs), n_val_docs=len(val_docs), smoke=SMOKE_TEST, phase1_pos_type=lm_cfg.pos_type, ckpt_kind=CKPT_KIND)
(RUN_DIR / "run_log.json").write_text(json.dumps(_jsonable(log), indent=2))
torch.save({"trainable": best["state"], "prefix_P0": model.prefix.P0.cpu(), "method": METHOD, "hp": HP[METHOD], "gen": GEN, "log": _jsonable(log)}, RUN_DIR / f"summ_{RUN_NAME}.pt")
print("saved:", sorted(p.name for p in RUN_DIR.iterdir()))

saved: ['best_trainable.pt', 'run_log.json', 'submission', 'summ_p6_dec01_prefix.pt', 'val_summaries.json']


## KD teacher export (for run #3)
Two signals, both keyed by `"<doc uid>|<chunk index>"` so the student notebook (same tokenizer, same chunking and alignment code) can match them exactly:
* **Word-level KD** (Hinton et al. 2015; Kim & Rush 2016): for every TR chunk pair with a target, the teacher's **top-k logits** at each gold target position (+ the log-sum-exp over the full vocabulary, so the student can recover true probabilities of the top-k tokens). Possible across architectures only because all Phase-1 models share one tokenizer.
* **Sequence-level KD** (Kim & Rush 2016): the teacher's own greedy summaries for up to `TEACHER_GEN_PAIRS` TR chunks (with the chosen decoding config). The student trains on them as targets.

In [26]:
# Cell 26 — Export teacher signals
if EXPORT_TEACHER and not SMOKE_TEST:
    t0 = time.time(); model.eval()
    tr_pairs = [(di, j) for di, d in enumerate(train_docs) for j, t in enumerate(d["tgts"]) if t is not None]
    topk = {}
    with torch.no_grad():
        for b in tqdm(make_microbatches(train_docs, tr_pairs, shuffle=False, budget=2 * MICRO_TOKENS), desc="teacher top-k", leave=False):
            x, y = build_batch(train_docs, b)
            with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
                h = backbone_hidden(model, x)
            for r, (di, j) in enumerate(b):
                sel = y[r] != -100
                lg = model.lm_head(h[r][sel].float()).float()
                v, ix = lg.topk(TEACHER_TOPK, dim=-1); lse = torch.logsumexp(lg, -1)
                topk[f"{train_docs[di]['uid']}|{j}"] = (v.half().cpu().numpy(), ix.to(torch.int32).cpu().numpy().astype(np.uint16),
                                                        lse.cpu().numpy().astype(np.float32))
    print(f"top-k logits for {len(topk)} pairs in {time.time() - t0:.0f}s")
    t1 = time.time(); gen_out = {}
    rng = random.Random(SEED); cand = list(tr_pairs); rng.shuffle(cand); cand = cand[:TEACHER_GEN_PAIRS]
    cand.sort(key=lambda p: len(prompt_ids(train_docs[p[0]], p[1])))
    for s in range(0, len(cand), GEN["batch"]):
        b = cand[s:s + GEN["batch"]]
        res = generate(model, [prompt_ids(train_docs[di], j) for di, j in b], GEN["max_new"], None, GEN["no_repeat_ngram"], GEN["rep_penalty"])
        for (di, j), o in zip(b, res): gen_out[f"{train_docs[di]['uid']}|{j}"] = np.asarray(o, dtype=np.uint16)
        if (time.time() - t1) / 60 > TEACHER_GEN_MIN:
            print(f"TEACHER_GEN_MIN reached after {len(gen_out)} chunks"); break
    print(f"teacher generations for {len(gen_out)} chunks in {time.time() - t1:.0f}s")
    meta = dict(teacher=RUN_NAME, method=METHOD, hp=HP[METHOD], chunk_tokens=CHUNK_TOKENS, tgt_max=TGT_MAX, min_align_f1=MIN_ALIGN_F1,
                max_full_chars=MAX_FULL_CHARS, tag_template=TAG_TEMPLATE, sep_text=SEP_TEXT, topk=TEACHER_TOPK, gen=GEN,
                params_total=int(model.n_total), params_trainable=int(model.n_trainable), val=val_res, val_checksum=VAL_CHECKSUM_EXPECTED)
    path = RUN_DIR / f"teacher_{RUN_NAME}.pt"
    torch.save(dict(meta=meta, topk=topk, gen=gen_out), path)
    print(f"saved {path} ({path.stat().st_size / 1e6:.0f} MB) | total {time.time() - t0:.0f}s")
else:
    print("teacher export skipped")

teacher export skipped


In [27]:
# Cell 27 — PASTE TO CLAUDE
short = dict(run=RUN_NAME, smoke=SMOKE_TEST, hp=HP[METHOD], params_trainable=model.n_trainable, params_total=model.n_total,
             history=history, best_epoch=best["epoch"], stats={k: (round(v, 4) if isinstance(v, float) else v) for k, v in stats.items()},
             gen=GEN, decode_sweep=sweep_res, val={k: round(v, 4) for k, v in val_res.items()},
             val_extra={k: v for k, v in val_extra.items() if k != "seconds"})
print("================ PASTE TO CLAUDE ================")
print(json.dumps(_jsonable(short), separators=(",", ":")))
print("=================================================")

================ PASTE TO CLAUDE ================
{"run":"p6_dec01_prefix","smoke":false,"hp":{"lr":0.002,"wd":0.0,"prefix_len":64,"mlp_hidden":128,"init_text":"The following is a summary of a judgment of the Supreme Court of India. It states the facts of the case, the questions of law, the arguments of the parties, the reasoning of the Court and its final decision. "},"params_trainable":1683584,"params_total":56615552,"history":[{"epoch":1,"train_loss":2.1386377018007003,"val_loss":2.0754008912288384,"seconds":1095.4491469860077,"steps":1459,"q_r1":0.5233037050440478,"q_r2":0.23873680794810395,"q_rL":0.2444766932877119},{"epoch":2,"train_loss":2.006799156943031,"val_loss":1.9857127031386688,"seconds":1099.6622660160065,"steps":2926,"q_r1":0.5276875280211899,"q_r2":0.24672559617623424,"q_rL":0.25097603616755576},{"epoch":3,"train_loss":1.947998130837466,"val_loss":1.9360386755349992,"seconds":1109.84303855896,"steps":4399,"q_r1":0.5279282469168448,"q_r2":0.249972189426752,"q_rL":0.2556